# 01 — Data Inspection and Inventory

## Objective

This notebook examines the original UNODC Individual Drug Seizures (IDS)
workbooks before database construction.

It assesses:
- Workbook and sheet structure
- Column consistency across years
- Data types and date coverage
- Missing values and nonpositive quantities
- Exact duplicate rows
- Country and drug coverage
- Drug reference information

The original Excel files are not modified.

In [1]:
# Imports and project paths

from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project root, whether Jupyter starts from the root
# directory or from the notebooks directory.
current_path = Path.cwd()

project_root = next(
    (
        path for path in [current_path, *current_path.parents]
        if (path / "data" / "raw").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/raw."
    )

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data folder:", raw_dir)
print("Processed data folder:", processed_dir)

Project root: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals
Raw data folder: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\data\raw
Processed data folder: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\data\processed


In [2]:
#Identify the workbooks
excel_files = sorted(raw_dir.glob("*.xlsx"))

if not excel_files:
    raise FileNotFoundError("No Excel workbooks found in data/raw.")

print(f"Number of workbooks: {len(excel_files)}\n")

for file_path in excel_files:
    size_mb = file_path.stat().st_size / 1_000_000
    print(f"{file_path.name} — {size_mb:.2f} MB")

Number of workbooks: 3

IDS-data-2011_17-May24.xlsx — 73.57 MB
IDS-data-2018-2022.xlsx — 65.36 MB
IDS-data-2023-2026.xlsx — 47.40 MB


In [3]:
# Inspect workbook sheets and columns
workbook_inventory = []

for file_path in excel_files:

    print(f"\n{'=' * 80}")
    print("FILE:", file_path.name)

    with pd.ExcelFile(file_path) as xls:

        print("Sheets:", xls.sheet_names)

        for sheet in xls.sheet_names:

            headers = pd.read_excel(
                xls,
                sheet_name=sheet,
                nrows=0
            ).columns.tolist()

            workbook_inventory.append({
                "file": file_path.name,
                "sheet": sheet,
                "number_of_columns": len(headers),
                "columns": headers
            })

            print(f"\nSheet: {sheet}")
            print(f"Number of columns: {len(headers)}")
            print(headers)

workbook_inventory_df = pd.DataFrame(workbook_inventory)


FILE: IDS-data-2011_17-May24.xlsx
Sheets: ['2011', '2012', '2013', '2014', '2015', '2016', '2017', 'Drug list and Measurement Units']

Sheet: 2011
Number of columns: 14
['Seizure Date', 'Country/Territory of Seizure', 'ISO3', 'Subregion', 'Region', 'Drug/Substance', 'Quantity Seized', 'Measurement Unit', 'City', 'Administrative Region', 'Physical Seizure Location', 'Trafficking Mode of Transportation', 'Source', 'Reporting Channel']

Sheet: 2012
Number of columns: 14
['Seizure Date', 'Country/Territory of Seizure', 'ISO3', 'Subregion', 'Region', 'Drug/Substance', 'Quantity Seized', 'Measurement Unit', 'City', 'Administrative Region', 'Physical Seizure Location', 'Trafficking Mode of Transportation', 'Source', 'Reporting Channel']

Sheet: 2013
Number of columns: 14
['Seizure Date', 'Country/Territory of Seizure', 'ISO3', 'Subregion', 'Region', 'Drug/Substance', 'Quantity Seized', 'Measurement Unit', 'City', 'Administrative Region', 'Physical Seizure Location', 'Trafficking Mode of Tran

In [4]:
# Inspect a representative annual sheet
sample_file = excel_files[0]

with pd.ExcelFile(sample_file) as xls:
    sample_year = next(
        sheet for sheet in xls.sheet_names if sheet.isdigit()
    )

    sample_df = pd.read_excel(
        xls,
        sheet_name=sample_year,
        nrows=5
    )

print("Sample workbook:", sample_file.name)
print("Sample year:", sample_year)
print("Columns:")
print(sample_df.columns.tolist())

display(sample_df)

Sample workbook: IDS-data-2011_17-May24.xlsx
Sample year: 2011
Columns:
['Seizure Date', 'Country/Territory of Seizure', 'ISO3', 'Subregion', 'Region', 'Drug/Substance', 'Quantity Seized', 'Measurement Unit', 'City', 'Administrative Region', 'Physical Seizure Location', 'Trafficking Mode of Transportation', 'Source', 'Reporting Channel']


,Seizure Date,Country/Territory of Seizure,ISO3,Subregion,Region,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Physical Seizure Location,Trafficking Mode of Transportation,Source,Reporting Channel
0,2011-12-31,Belgium,BEL,Western and Central Europe,Europe,Non-specified cocaine-type,2.00,kg,Brussels,NaN,Airport,Airplane,Government,Individual Drug Seizures (IDS)
1,2011-12-31,Bolivia (Plurinational State of),BOL,South America,Americas,Cocaine base,1.20,g,La Paz,NaN,Data not available,Data not available,Government,Individual Drug Seizures (IDS)
2,2011-12-31,Bolivia (Plurinational State of),BOL,South America,Americas,Cocaine base,2837.00,g,Cochabamba,NaN,Data not available,Data not available,Government,Individual Drug Seizures (IDS)
3,2011-12-31,Bolivia (Plurinational State of),BOL,South America,Americas,"Cocaine hydrochloride (HCl, powder cocaine)",4905.00,g,Cochabamba,NaN,Data not available,Data not available,Government,Individual Drug Seizures (IDS)
4,2011-12-31,Colombia,COL,South America,Americas,Cannabis herb (marijuana),2.33,kg,Bogota D.C.,NaN,Road/ Street/highway,Car/motorcycle,Government,Individual Drug Seizures (IDS)


In [5]:
# Check column consistency
def standardize_columns(columns):
    return [
        str(column).strip().replace("Subrgion", "Subregion")
        for column in columns
    ]


annual_sheets = [
    item for item in workbook_inventory
    if item["sheet"].isdigit()
]

reference_columns = set(
    standardize_columns(annual_sheets[0]["columns"])
)

column_comparison = []

for item in annual_sheets:

    current_columns = set(
        standardize_columns(item["columns"])
    )

    column_comparison.append({
        "year": int(item["sheet"]),
        "file": item["file"],
        "same_columns": current_columns == reference_columns,
        "missing_columns": sorted(reference_columns - current_columns),
        "additional_columns": sorted(current_columns - reference_columns)
    })

column_comparison_df = pd.DataFrame(column_comparison)

display(column_comparison_df)

if not column_comparison_df["same_columns"].all():
    print("Warning: Some annual sheets have different columns.")
else:
    print("All annual sheets have the same columns after standardization.")

,year,file,same_columns,missing_columns,additional_columns
0,2011,IDS-data-2011_17-May24.xlsx,True,[],[]
1,2012,IDS-data-2011_17-May24.xlsx,True,[],[]
2,2013,IDS-data-2011_17-May24.xlsx,True,[],[]
3,2014,IDS-data-2011_17-May24.xlsx,True,[],[]
4,2015,IDS-data-2011_17-May24.xlsx,True,[],[]
5,2016,IDS-data-2011_17-May24.xlsx,True,[],[]
6,2017,IDS-data-2011_17-May24.xlsx,True,[],[]
7,2018,IDS-data-2018-2022.xlsx,True,[],[]
8,2019,IDS-data-2018-2022.xlsx,True,[],[]
9,2020,IDS-data-2018-2022.xlsx,True,[],[]


All annual sheets have the same columns after standardization.


In [6]:
# Build the annual data inventory
START_YEAR = 2011
END_YEAR = 2026

inventory = []

required_columns = {
    "Seizure Date",
    "ISO3",
    "Drug/Substance",
    "Quantity Seized",
    "Measurement Unit"
}

for file_path in excel_files:

    with pd.ExcelFile(file_path) as xls:

        for sheet in xls.sheet_names:

            if not sheet.isdigit():
                continue

            year = int(sheet)

            if not START_YEAR <= year <= END_YEAR:
                continue

            print(f"Inspecting {year}...")

            data = pd.read_excel(
                xls,
                sheet_name=sheet
            )

            # Standardize column names
            data.columns = standardize_columns(data.columns)

            missing_columns = required_columns - set(data.columns)

            if missing_columns:
                raise ValueError(
                    f"{year}: Missing required columns: {missing_columns}"
                )

            dates = pd.to_datetime(
                data["Seizure Date"],
                errors="coerce"
            )

            quantity = pd.to_numeric(
                data["Quantity Seized"],
                errors="coerce"
            )

            inventory.append({
                "year": year,
                "file": file_path.name,
                "rows": len(data),
                "columns": len(data.columns),

                "date_min": dates.min(),
                "date_max": dates.max(),
                "unique_dates": dates.nunique(),
                "missing_dates": dates.isna().sum(),

                "missing_countries": data["ISO3"].isna().sum(),
                "missing_drugs": data["Drug/Substance"].isna().sum(),
                "missing_quantities": quantity.isna().sum(),

                "nonpositive_quantities": quantity.le(0).sum(),
                "negative_quantities": quantity.lt(0).sum(),
                "zero_quantities": quantity.eq(0).sum(),

                "exact_duplicate_rows": data.duplicated().sum(),

                "unique_drugs": data["Drug/Substance"].nunique(),
                "unique_countries": data["ISO3"].nunique(),

                "missing_transportation": (
                    data["Trafficking Mode of Transportation"].isna().sum()
                ),
                "transportation_not_available": (
                    data["Trafficking Mode of Transportation"]
                    .eq("Data not available").sum()
                )
            })

            del data

inventory_df = (
    pd.DataFrame(inventory)
    .sort_values("year")
    .reset_index(drop=True)
)

display(inventory_df)

Inspecting 2011...
Inspecting 2012...
Inspecting 2013...
Inspecting 2014...
Inspecting 2015...
Inspecting 2016...
Inspecting 2017...
Inspecting 2018...
Inspecting 2019...
Inspecting 2020...
Inspecting 2021...
Inspecting 2022...
Inspecting 2023...
Inspecting 2024...
Inspecting 2025...
Inspecting 2026...


,year,file,rows,columns,date_min,date_max,unique_dates,missing_dates,missing_countries,missing_drugs,missing_quantities,nonpositive_quantities,negative_quantities,zero_quantities,exact_duplicate_rows,unique_drugs,unique_countries,missing_transportation,transportation_not_available
0,2011,IDS-data-2011_17-May24.xlsx,41806,14,2011-01-01,2011-12-31,365,0,0,0,0,3,0,3,2465,102,67,0,27543
1,2012,IDS-data-2011_17-May24.xlsx,62090,14,2012-01-01,2012-12-31,366,0,0,0,0,3,0,3,4528,143,72,0,52471
2,2013,IDS-data-2011_17-May24.xlsx,62036,14,2013-01-01,2013-12-31,365,0,0,0,0,58,0,58,4999,139,63,0,53121
3,2014,IDS-data-2011_17-May24.xlsx,265457,14,2014-01-01,2014-12-31,365,0,0,0,0,0,0,0,40496,177,53,0,259419
4,2015,IDS-data-2011_17-May24.xlsx,238294,14,2015-01-01,2015-12-31,365,0,0,0,0,2,0,2,40117,142,45,0,234456
5,2016,IDS-data-2011_17-May24.xlsx,307817,14,2016-01-01,2016-12-31,366,0,0,0,0,6,0,6,53697,147,46,0,268107
6,2017,IDS-data-2011_17-May24.xlsx,232409,14,2017-01-01,2017-12-31,365,0,0,0,0,0,0,0,24112,109,54,0,228756
7,2018,IDS-data-2018-2022.xlsx,220003,14,2018-01-01,2018-12-31,365,0,0,0,0,0,0,0,15424,112,82,0,210300
8,2019,IDS-data-2018-2022.xlsx,119368,14,2019-01-01,2019-12-31,365,0,0,0,0,7,0,7,5918,106,77,0,105437
9,2020,IDS-data-2018-2022.xlsx,151682,14,2020-01-01,2020-12-31,366,0,0,0,0,7,0,7,5026,120,84,0,132335


In [7]:
# Overall dataset summary
total_records = inventory_df["rows"].sum()

print(f"Total records: {total_records:,}")
print(f"Years covered: {inventory_df['year'].min()}–{inventory_df['year'].max()}")

print(
    "First date:",
    inventory_df["date_min"].min()
)

print(
    "Last date:",
    inventory_df["date_max"].max()
)

print(
    "Total exact duplicate rows:",
    f"{inventory_df['exact_duplicate_rows'].sum():,}"
)

print(
    "Total zero quantities:",
    f"{inventory_df['zero_quantities'].sum():,}"
)

print(
    "Total negative quantities:",
    f"{inventory_df['negative_quantities'].sum():,}"
)

Total records: 2,841,288
Years covered: 2011–2026
First date: 2011-01-01 00:00:00
Last date: 2026-06-30 00:00:00
Total exact duplicate rows: 239,657
Total zero quantities: 142
Total negative quantities: 0


In [8]:
# Inspect the drug reference sheet
REFERENCE_SHEET = "Drug list and Measurement Units"

reference_file = excel_files[0]

drug_reference = pd.read_excel(
    reference_file,
    sheet_name=REFERENCE_SHEET
)

drug_reference.columns = (
    drug_reference.columns
    .str.strip()
)

print("Reference file:", reference_file.name)
print("Rows:", len(drug_reference))
print("Columns:", drug_reference.columns.tolist())

display(drug_reference.head(20))

Reference file: IDS-data-2011_17-May24.xlsx
Rows: 184
Columns: ['Drug class/ group', 'Drug Name/Type', 'Drug Subgroup', 'Measurement  Unit']


,Drug class/ group,Drug Name/Type,Drug Subgroup,Measurement Unit
0,Amphetamine-type stimulants (excluding ecstasy),Amphetamine,NaN,plant
1,Amphetamine-type stimulants (excluding ecstasy),Captagon,NaN,tablet/pill
2,Amphetamine-type stimulants (excluding ecstasy),Fenethylline,NaN,unit
3,Amphetamine-type stimulants (excluding ecstasy),Methamphetamine,NaN,g
4,Amphetamine-type stimulants (excluding ecstasy),Methcathinone,NaN,kg
5,Amphetamine-type stimulants (excluding ecstasy),Non-specified amphetamine-type stimulants (exc...,NaN,lt
6,Amphetamine-type stimulants (excluding ecstasy),Other amphetamine-type stimulants (excluding e...,Other amphetamine-type stimulants (excluding e...,"other, specify"
7,Cannabis-type (excluding synthetic cannabinoids),Cannabis herb (marijuana),NaN,NaN
8,Cannabis-type (excluding synthetic cannabinoids),Cannabis oil,NaN,NaN
9,Cannabis-type (excluding synthetic cannabinoids),Cannabis plants,NaN,NaN


In [9]:
# Drug reference coverage
# Identify all substances appearing in the annual source data
main_drugs = set()

for file_path in excel_files:

    with pd.ExcelFile(file_path) as xls:

        for sheet in xls.sheet_names:

            if not sheet.isdigit():
                continue

            year = int(sheet)

            if not START_YEAR <= year <= END_YEAR:
                continue

            data = pd.read_excel(
                xls,
                sheet_name=sheet,
                usecols=["Drug/Substance"]
            )

            main_drugs.update(
                data["Drug/Substance"].dropna().unique()
            )

reference_drugs = set(
    drug_reference["Drug Name/Type"].dropna().unique()
)

missing_from_reference = sorted(main_drugs - reference_drugs)
not_in_main_dataset = sorted(reference_drugs - main_drugs)

print("Substances in main dataset:", len(main_drugs))
print("Substances in reference:", len(reference_drugs))
print("Substances missing from reference:", len(missing_from_reference))
print("Reference substances absent from main dataset:", len(not_in_main_dataset))

display(pd.DataFrame({
    "substances_missing_from_reference": pd.Series(missing_from_reference)
}))

display(pd.DataFrame({
    "reference_substances_not_in_main_dataset": pd.Series(not_in_main_dataset)
}))

Substances in main dataset: 347
Substances in reference: 184
Substances missing from reference: 190
Reference substances absent from main dataset: 27


,substances_missing_from_reference
0,PB-22
1,1 benzyl N (quinolin 8 yl) 1H indazole 3 carbo...
2,2-Aminoindane
3,2-Fluoromethamphetamine
4,2-MMC
...,...
185,bk_MDEA
186,naphthalen-1-yl-1-benzyl-1H-indole-3-carboxyla...
187,para-Methoxy-N-methylamphetamine
188,para-Methoxyamphetamine


,reference_substances_not_in_main_dataset
0,5-Aminoisotonitazene
1,5-methyl etodesnitazene
2,6-methyl etodesnitazene
3,Activated Carbon
4,Benzyl Chloride
5,Butonitazene
6,Calcium Oxide
7,Calcium carbonate
8,Etazene
9,Ethyl acetate


In [10]:
# Check reference quality
print(
    "Duplicated drug names:",
    drug_reference["Drug Name/Type"].duplicated().sum()
)

print("\nMissing values in reference:")
display(
    drug_reference.isna().sum().to_frame("missing_values")
)

print("\nUnique drug classes:")
print(drug_reference["Drug class/ group"].nunique())

print("\nUnique drug subgroups:")
print(drug_reference["Drug Subgroup"].nunique())

Duplicated drug names: 0

Missing values in reference:


,missing_values
Drug class/ group,0
Drug Name/Type,0
Drug Subgroup,104
Measurement Unit,177



Unique drug classes:
13

Unique drug subgroups:
20


In [11]:
# Save inventory and reference outputs
inventory_path = processed_dir / "annual_data_inventory.csv"
reference_path = processed_dir / "drug_reference.csv"

inventory_df.to_csv(
    inventory_path,
    index=False
)

drug_reference.to_csv(
    reference_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(inventory_path)
print(reference_path)

Saved:
C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\data\processed\annual_data_inventory.csv
C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\data\processed\drug_reference.csv


In [12]:
# Consolidate all annual sheets
annual_data = []

for file in excel_files:
    xls = pd.ExcelFile(file)

    for sheet in xls.sheet_names:
        if not sheet.isdigit():
            continue

        year = int(sheet)

        if not START_YEAR <= year <= END_YEAR:
            continue

        df = pd.read_excel(file, sheet_name=sheet)

        # Standardize column names
        df.columns = standardize_columns(df.columns)

        # Add source metadata for traceability
        df["source_year"] = year
        df["source_file"] = file.name
        df["source_sheet"] = sheet

        annual_data.append(df)

# Combine all years
ids = pd.concat(annual_data, ignore_index=True)

# Parse dates and quantities
ids["Seizure Date"] = pd.to_datetime(
    ids["Seizure Date"],
    errors="coerce"
)

ids["Quantity Seized"] = pd.to_numeric(
    ids["Quantity Seized"],
    errors="coerce"
)

# Remove leading/trailing spaces from text fields
text_cols = ids.select_dtypes(
    include=["object", "string"]
).columns

for col in text_cols:
    ids[col] = ids[col].str.strip()

# Save the consolidated dataset
output_path = processed_dir / "ids_2011_2026.parquet"

ids.to_parquet(
    output_path,
    index=False
)

print("Saved to:", output_path)
print("Rows:", len(ids))
print("Columns:", len(ids.columns))
print("Date range:", ids["Seizure Date"].min(), "-", ids["Seizure Date"].max())

# Verify metadata
print("\nMetadata verification:")
display(ids[["source_year", "source_file", "source_sheet"]].head())

print("\nRecords by source year:")
display(ids.groupby("source_year").size().reset_index(name="records"))


Saved to: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\data\processed\ids_2011_2026.parquet
Rows: 2841288
Columns: 17
Date range: 2011-01-01 00:00:00 - 2026-06-30 00:00:00

Metadata verification:


,source_year,source_file,source_sheet
0,2011,IDS-data-2011_17-May24.xlsx,2011
1,2011,IDS-data-2011_17-May24.xlsx,2011
2,2011,IDS-data-2011_17-May24.xlsx,2011
3,2011,IDS-data-2011_17-May24.xlsx,2011
4,2011,IDS-data-2011_17-May24.xlsx,2011



Records by source year:


,source_year,records
0,2011,41806
1,2012,62090
2,2013,62036
3,2014,265457
4,2015,238294
5,2016,307817
6,2017,232409
7,2018,220003
8,2019,119368
9,2020,151682


## Conclusions

- The IDS dataset is distributed across three Excel workbooks, covering 2011–2026.
- Annual sheets contain 14 columns, although some column labels require standardization.
- The 2026 data are partial and currently cover January–June.
- The annual inventory provides a consistent overview of record counts, missing values, quantities, duplicate rows, and reporting coverage.
- The drug reference sheet contains fewer substances than the main dataset. Unmatched substances should remain in the analytical dataset rather than being discarded.
- Exact duplicate rows are identified for investigation, but are not automatically removed because the source does not provide a unique seizure identifier.